# Run the AEROBAT research pipeline

This is the only execution notebook. It runs Stage 1 hypothesis generation, Stage 2 matched-configuration design, Stage 3 matched simulation runs, and Stage 4 blind review and behavior scoring. The research manager applies the ranking, coherence, and fidelity gates; final research reports are optional. Existing artifacts are loaded unless `FORCE_STAGE` is set. Stage, cache, hypothesis, simulation, round, retry, and review progress is logged in the cell output by default.

In [ ]:
from pathlib import Path
import logging
import sys

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))

# Show AEROBAT's stage, hypothesis, simulation, round, retry, and cache progress by default.
progress_logger = logging.getLogger('aerobat')
for existing_handler in list(progress_logger.handlers):
    if getattr(existing_handler, '_aerobat_notebook_progress', False):
        progress_logger.removeHandler(existing_handler)
progress_handler = logging.StreamHandler(sys.stdout)
progress_handler._aerobat_notebook_progress = True
progress_handler.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s', datefmt='%H:%M:%S'))
progress_logger.addHandler(progress_handler)
progress_logger.setLevel(logging.INFO)
progress_logger.propagate = False

from aerobat import AerobatPipeline, load_config

CONFIG_PATH = ROOT / 'seed.yaml'
config = load_config(CONFIG_PATH)
pipeline = AerobatPipeline(config)
print(f'Target behavior Y: {config.target_behavior}')
print(f'Results:           {config.target_behavior_dir}')

In [ ]:
# None runs the complete cache-aware pipeline. Otherwise, list the stages to rerun.
# Earlier prerequisites load from cache (or run if missing); later stages are skipped.
STAGES = [1,2,3,4]  # Examples: [4] for review/analysis only, or [3, 4] for simulation onward.
GENERATE_RESEARCH_REPORTS = True
run = await pipeline.run(stages=STAGES, generate_reports=GENERATE_RESEARCH_REPORTS)

print(f'Hypotheses H generated:       {len(run.hypothesis_generation["hypotheses"])}')
print(f'Hypotheses tested:            {len(run.matched_configurations)}')
print(f'Matched simulation runs S_ij: {sum(len(x["runs"]) for x in run.matched_simulation_runs.values())}')
print(f'Blind reviews / scores:       {sum(int(x.get("successful_count", 0)) for x in run.blind_reviews.values())}')
print(f'Research reports:             {len(run.research_reports)}')

In [ ]:
# Compact hypothesis-level result view
import pandas as pd

pd.DataFrame([
    {
        'hypothesis_id': hypothesis_id,
        'hypothesized_causal_variable_X': result['variable'],
        'n_observations': result['quantitative_analysis']['aggregate_mean']['n'],
        'BF10': result['quantitative_analysis']['aggregate_mean']['monotone_analysis']['bf10'],
        'Delta': result['quantitative_analysis']['aggregate_mean']['effect_size']['Delta'],
        'effect_class': {'positive': 'Positive', 'negative': 'Negative', 'no_effect': 'No effect', 'inconclusive': 'Inconclusive', 'direction_unresolved': 'Inconclusive'}.get(result['quantitative_analysis']['aggregate_mean']['effect_class']),
    }
    for hypothesis_id, result in run.statistical_analyses.items()
]).sort_values('BF10', ascending=False)